<a href="https://colab.research.google.com/github/sjayavelu73/langgraph/blob/lang1/mature_pdf_ingestion_ensemble_retriever_query_routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install -U langchain_core
#!pip install -U langchain_community
#!pip install -U langchain_chroma
#!pip install -U langchain_openai
#!pip install PyMUPDF
#!pip install pdfplumber
#!pip install pdf2image
#!pip install rank_bm25
#!pip install pytesseract
#!pip install langgraph
#!pip install langdetect

from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough , RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage,HumanMessage ,BaseMessage , ToolMessage,SystemMessage
from langchain import hub
from langchain_core.tools import tool
from langgraph.graph import START,END ,StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from typing import TypedDict , Union , Annotated , Sequence
from google.colab import drive
from google.colab import userdata
import fitz
import pdfplumber
import pdf2image
from langchain_community.retrievers.bm25 import BM25Retriever
import os,time,re,shutil ,io
from PIL import Image
import pytesseract
from langdetect import detect, DetectorFactory, LangDetectException

# Mount the drive
drive.mount('/content/drive')
file_paths=['/content/drive/MyDrive/RAG.pdf','/content/drive/MyDrive/deeplearning_toolkit.pdf','/content/drive/MyDrive/transformers.pdf','/content/drive/MyDrive/attention_is_what_you_need.pdf']

# Initialize OpenAI
os.environ['OPENAI_API_KEY'] = userdata.get('ai_agents_openai')
model_without_tools = ChatOpenAI(model='gpt-3.5-turbo',temperature=0)

# Heuristic function to detect garbled text
def is_likely_garbled_text(text):
    if not text or len(text) < 100:
        return True
    alpha_count = sum(c.isalnum() for c in text)
    non_alpha_count = sum(not c.isalnum() for c in text)
    ratio = alpha_count / non_alpha_count if non_alpha_count else 1.0
    if ratio < 0.1:
        return True
    control_chars = len(re.findall(r"[\x00-\x1F\x7F]", text))
    if control_chars > 10:
        return True
    return False

# Process Files
pdf_splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=200)
all_docs=[]
for file_path in file_paths:
  print(file_path)
  method_used = "PyMuPDF"
  print(method_used)
  pdfdoc=fitz.open(file_path)
  all_page_text=[]
  for page in pdfdoc:
    pdf_text=page.get_text()
    page_num = page.number
    if is_likely_garbled(pdf_text):
      #pdfdoc.close()
      print("using pdfplumber as PyMUPDF is not able to generate optimal text")
      pdfname=pdfplumber.open(file_path)
      pdf_text=pdfname.pages[page_num].extract_text() or ""
      if is_likely_garbled(pdf_text):
        print("using OCR as Pdfplumber is not able to generate optimal text")
        pix = page.get_pixmap(dpi=300)
        img_bytes = pix.pil_tobytes(format="PNG")
        img_obj = Image.open(io.BytesIO(img_bytes))
        pdf_text = pytesseract.image_to_string(img_obj)
    chunks=pdf_splitter.split_text(pdf_text)
    docs = [
            Document(page_content=chunk, metadata={"page": page_num+1, "source": file_path.split("/")[-1]})
            for chunk in chunks if chunk.strip()
           ]
    all_docs.extend(docs)









Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/RAG.pdf
PyMuPDF
using pdfplumber as PyMUPDF is not able to generate optimal text
using OCR as Pdfplumber is not able to generate optimal text
using pdfplumber as PyMUPDF is not able to generate optimal text
using OCR as Pdfplumber is not able to generate optimal text
using pdfplumber as PyMUPDF is not able to generate optimal text
using pdfplumber as PyMUPDF is not able to generate optimal text
using OCR as Pdfplumber is not able to generate optimal text
using pdfplumber as PyMUPDF is not able to generate optimal text
using pdfplumber as PyMUPDF is not able to generate optimal text
using OCR as Pdfplumber is not able to generate optimal text
using pdfplumber as PyMUPDF is not able to generate optimal text
using pdfplumber as PyMUPDF is not able to generate optimal text
using pdfplumber as PyMUPDF is not able to generate optimal text
usi

In [38]:
embeddings=OpenAIEmbeddings()
persist_dir='/content/drive/MyDrive/MYRAG4'
collection_name='MYRAG4'
from chromadb.api.client import SharedSystemClient
from langchain_core.output_parsers import StrOutputParser

SharedSystemClient.clear_system_cache()
from langchain.load import dumps, loads
persist_dir='/content/drive/MyDrive/MYRAG5'
collection_name='MYRAG5'
import json
# Step 1: Close or clean up any open DB objects if any
# (In Python, usually just deleting references is enough)

# Step 2: Remove directory safely
if os.path.exists(persist_dir):
    print("Directory exists")
    try:
        shutil.rmtree(persist_dir)
    except PermissionError:
        # On some filesystems permission errors happen, try again after a delay
        time.sleep(1)
        shutil.rmtree(persist_dir)

    # Give the filesystem some time to flush deletes
    time.sleep(2)

# Step 3: Confirm directory is removed before proceeding
assert not os.path.exists(persist_dir), "Persist directory still exists after deletion!"

db=Chroma.from_documents(all_docs,embeddings,persist_directory=persist_dir,collection_name=collection_name)
retriever=db.as_retriever(kwargs={"k":5})
query = "what is an function"

retriever.invoke(query)

prompt=hub.pull("rlm/rag-prompt")

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain=({"context": retriever | format_docs , "question": RunnablePassthrough()} | prompt | model_without_tools | StrOutputParser())
query = "Describe an attention function"

response=rag_chain.invoke(query)
print(response)

print("------------------------------------------------------------------")

#bm25 retriever

bm25_retriever = BM25Retriever.from_documents(all_docs)
bm25_retriever.k = 5
bm25_rag_chain=({"context": bm25_retriever | format_docs , "question": RunnablePassthrough()} | prompt | model_without_tools | StrOutputParser())
query = "Describe an attention function"

response=bm25_rag_chain.invoke(query)
print(response)




Directory exists
An attention function maps a query and key-value pairs to an output using vectors, computing a weighted sum. Self-attention, or intra-attention, relates different positions in a sequence to compute a representation. Attention mechanisms are integral in sequence modeling and transduction tasks, allowing for modeling dependencies without regard to distance.
------------------------------------------------------------------
An attention function maps a query and key-value pairs to an output using vectors, computed as a weighted sum. The two most common types are additive attention and dot-product attention. Dot-product attention is faster and more space-efficient due to optimized matrix multiplication.


In [42]:
from langchain.retrievers.ensemble import EnsembleRetriever
retrievers = [retriever, bm25_retriever]
weights = [0.7,0.3]
ensemble_retriever = EnsembleRetriever(
    retrievers=retrievers, weights=weights
)

ensemble_chain= ({"context":ensemble_retriever , "question": RunnablePassthrough()} | prompt | model_without_tools | StrOutputParser())
ensemble_chain.invoke(query)


'An attention function maps a query and key-value pairs to an output using vectors. It computes a weighted sum of the values based on the compatibility function of the query with the corresponding key. Attention mechanisms are integral in sequence modeling and transduction tasks, allowing for modeling dependencies without considering their distance in input or output sequences. The Transformer model architecture relies entirely on an attention mechanism to draw global dependencies between input and output, enabling more parallelization and achieving state-of-the-art results in translation quality.'

In [43]:
template = """You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines. Original question: {question}"""

prompt_perspectives=ChatPromptTemplate.from_template(template)
from langchain.load import dumps, loads

generate_queries = (
    prompt_perspectives
    | ChatOpenAI(temperature=0)
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

# Retrieve
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":query})
print(docs)


[Document(id='43ddd5ed-5a7e-4764-ac24-b066a875fdad', metadata={'source': 'attention_is_what_you_need.pdf', 'page': 3}, page_content='3.2 ttention\n\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum'), Document(id='bf4a2f00-14a5-4a0c-a0d5-9138b08a1a54', metadata={'source': 'attention_is_what_you_need.pdf', 'page': 2}, page_content='Attention mechanisms have become an integral part of compelling sequence modeling and transduc-\ntion models in various tasks, allowing modeling of dependencies without regard to their distance in\nthe input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms\nare used in conjunction with a recurrent network.\n\nIn this work we propose the Transformer, a model architecture eschewing recurrence and instead\nrelying entirely on an attention mechanism to draw global depen

In [45]:
generated_queries = generate_queries.invoke({"question": query})
print("Generated Queries:")
for i, q in enumerate(generated_queries):
    print(f"{i+1}: {q}")

# Step 2: Retrieve and deduplicate documents based on these generated queries
retrieved_docs = ensemble_retriever.map().invoke(generated_queries)
unique_docs = get_unique_union(retrieved_docs)
print("\nRetrieved Unique Documents:")
for doc in unique_docs:
    print(f"Doc ID: {doc.id} - Source: {doc.metadata.get('source', 'unknown')}")

# Step 3: Format docs for the prompt input
context_text = format_docs(unique_docs)

# Step 4: Run the final prompt + model to get combined answer
inputs = {
    "context": context_text,
    "question": query
}
final_response = (prompt | model_without_tools | StrOutputParser()).invoke(inputs)

print("\nFinal Combined Answer:")
print(final_response)

Generated Queries:
1: 1. What is the role of an attention function in a neural network?
2: 2. How does an attention function enhance the performance of machine learning models?
3: 3. Can you explain the concept of an attention function and its significance in natural language processing?
4: 4. In what ways does an attention function improve the efficiency of deep learning algorithms?
5: 5. What are the key components of an attention function and how do they contribute to model interpretability?

Retrieved Unique Documents:
Doc ID: 18098e92-02b7-4afc-9137-60e4c67cc505 - Source: attention_is_what_you_need.pdf
Doc ID: 43ddd5ed-5a7e-4764-ac24-b066a875fdad - Source: attention_is_what_you_need.pdf
Doc ID: bf4a2f00-14a5-4a0c-a0d5-9138b08a1a54 - Source: attention_is_what_you_need.pdf
Doc ID: e11ea4cd-8b51-48cc-86e4-3662f0c63b9a - Source: attention_is_what_you_need.pdf
Doc ID: 18098e92-02b7-4afc-9137-60e4c67cc505 - Source: attention_is_what_you_need.pdf
Doc ID: 43ddd5ed-5a7e-4764-ac24-b066a875f

In [52]:
# Maybe a more mature approach

# Step 3: Build the RAG chain integrating expanded queries, retrieval, dedup, formatting, and generation
rag_chain = ({
    "context": generate_queries | retriever.map() | get_unique_union | format_docs,
    "question": RunnablePassthrough()  # pass the original question unchanged
} | prompt | model_without_tools | StrOutputParser())


# Step 4: Run the chain on an input question
query = "Describe an attention function?"
response = rag_chain.invoke({"question": query})

print(response)

An attention function maps a query and key-value pairs to an output using vectors. It allows modeling dependencies without regard to their distance in sequences. The Transformer model relies entirely on an attention mechanism to draw global dependencies between input and output.
